# 02 — Feature engineering

How the 375-column feature matrix is built: application-level ratios, and streaming aggregates over the large history tables.

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
# run from repo root so `src` and configs resolve
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
pd.set_option("display.max_columns", 60)

## Application ratios

Interpretable affordability ratios and combinations of the external scores.

In [2]:
from src.data.preprocessing import clean_application
from src.features.feature_engineering import application_features
from src.data.load_data import load_config, load_application
cfg = load_config()
app = load_application(cfg, 'train')
eng = application_features(clean_application(app))
new_cols = [c for c in eng.columns if c not in app.columns]
print('added', len(new_cols), 'application features:')
print(new_cols)

added 15 application features:
['DAYS_EMPLOYED_ANOM', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_ANNUITY_RATIO', 'CREDIT_GOODS_RATIO', 'INCOME_PER_PERSON', 'INCOME_PER_CHILD', 'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_TO_AGE_RATIO', 'EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'EXT_SOURCE_MAX', 'EXT_SOURCE_STD', 'EXT_SOURCE_PROD']


In [3]:
eng[['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','CREDIT_ANNUITY_RATIO',
     'EXT_SOURCE_MEAN','AGE_YEARS']].describe()

,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_ANNUITY_RATIO,EXT_SOURCE_MEAN,AGE_YEARS
count,307511.000000,307499.000000,307499.000000,307339.000000,307511.000000
mean,3.957948,0.180951,21.612320,0.509251,43.906898
std,2.689355,0.094553,7.823823,0.149802,11.947950
min,0.083333,0.007500,8.036674,0.000006,20.503765
25%,2.018667,0.114800,15.614496,0.413648,33.984943
50%,3.265067,0.162833,19.999998,0.524502,43.121151
75%,5.159880,0.229080,27.099983,0.622819,53.886379
max,84.736839,1.875965,45.305077,0.878903,69.073235


## Streaming aggregation over history tables

The history tables are large (bureau_balance is ~27M rows), so aggregates are computed in a two-pass streaming way that keeps memory proportional to the number of groups, not rows. The maths is unit-tested against a plain groupby in `tests/test_features.py`.

In [4]:
from src.features.feature_engineering import ChunkedAggregator
# tiny demo of the aggregator
demo = pd.DataFrame({'SK_ID_CURR':[1,1,2,2,2],'AMT':[10.,20.,3.,4.,5.]})
agg = ChunkedAggregator(key='SK_ID_CURR', value_cols=['AMT'])
agg.update(demo)
agg.result(prefix='DEMO_')

,SK_ID_CURR,DEMO_AMT_MEAN,DEMO_AMT_STD,DEMO_AMT_MIN,DEMO_AMT_MAX,DEMO_AMT_SUM,DEMO_COUNT
0,1,15.0,5.000000,10.0,20.0,30.0,2.0
1,2,4.0,0.816497,3.0,5.0,12.0,3.0


## The built matrix

Run `python main.py features` to build the full matrices; here we just load the metadata it saved.

In [5]:
meta = json.load(open('reports/tables/feature_meta.json'))
meta

{'n_train_rows': 307511,
 'n_test_rows': 48744,
 'n_application_features': 135,
 'n_total_features': 375,
 'target_positive_rate': 0.08072881945686496,
 'seconds': 69.8}

In [6]:
# if the matrix has been built, peek at it
import os
p = 'data/processed/train.parquet'
if os.path.exists(p):
    tr = pd.read_parquet(p)
    print('matrix shape:', tr.shape)
    print('example aggregate columns:', [c for c in tr.columns if c.startswith(('BURO_','PREV_','POS_','CC_'))][:10])
else:
    print('run: python main.py features')

matrix shape: (307511, 377)
example aggregate columns: ['BURO_DAYS_CREDIT_MEAN', 'BURO_DAYS_CREDIT_STD', 'BURO_DAYS_CREDIT_MIN', 'BURO_DAYS_CREDIT_MAX', 'BURO_DAYS_CREDIT_SUM', 'BURO_CREDIT_DAY_OVERDUE_MEAN', 'BURO_CREDIT_DAY_OVERDUE_STD', 'BURO_CREDIT_DAY_OVERDUE_MIN', 'BURO_CREDIT_DAY_OVERDUE_MAX', 'BURO_CREDIT_DAY_OVERDUE_SUM']
